# Research Hub demo

Runs the three README requests (A: chat-only, B: saved-file, C: weak-sourcing HITL gate).
Offline by default (`USE_LIVE_LLM = False`, keyless demo mode). Set `USE_LIVE_LLM = True`
and export `ANTHROPIC_API_KEY` to run it against the real pipeline instead.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../backend'))

USE_LIVE_LLM = False
if not USE_LIVE_LLM:
    os.environ.pop('ANTHROPIC_API_KEY', None)
    os.environ.pop('TAVILY_API_KEY', None)

from app.pipeline import research_pipeline
from app.tracing import find_trace_insight
from app.state import new_thread_id

## Request A — quick chat, no file, no human checkpoint

In [ ]:
thread_a = new_thread_id()
state_a = research_pipeline.invoke(
    {"thread_id": thread_a, "query": "What's new in the Model Context Protocol?"},
    config={"configurable": {"thread_id": thread_a}},
)
print(state_a.status)
print(state_a.final_report[:500])
print(find_trace_insight(state_a))

## Request B — full pipeline, ends with a saved file (new path, no confirmation)

In [ ]:
thread_b = new_thread_id()
state_b = research_pipeline.invoke(
    {
        "thread_id": thread_b,
        "query": "Research the top 3 vector databases and save a comparison to reports/vector-dbs.md.",
    },
    config={"configurable": {"thread_id": thread_b}},
)
print(state_b.status, state_b.output_path)
print(find_trace_insight(state_b))

## Request C — weak sourcing: reviewer requests a revision, then a human confirms
before the low-confidence answer is delivered.

In [ ]:
thread_c = new_thread_id()
state_c = research_pipeline.invoke(
    {"thread_id": thread_c, "query": "Is this claim about drug X still accurate?"},
    config={"configurable": {"thread_id": thread_c}},
)
print(state_c.status)  # likely 'interrupted' awaiting human confirmation

if state_c.status == "interrupted":
    state_c = research_pipeline.invoke(
        {"thread_id": thread_c, "resume": {"confirmed": True}},
        config={"configurable": {"thread_id": thread_c}},
    )

print(state_c.status)
print(find_trace_insight(state_c))